# Sentinel — a cost-aware payment fraud risk engine

**Track 02 (AI Risk Manager) · Razorpay Buildathon**

This notebook runs the whole thing on the real **IEEE-CIS Fraud Detection** data:
load → leakage-free time split → train a gradient-boosted model → calibrate →
choose *allow / review / block* thresholds that **minimise money lost** (under a
realistic manual-review capacity cap) → show the curves → score live transactions
with human-readable reasons.

> The metric that matters here is **money**, not accuracy. When fraud is a few percent
> of traffic, an accuracy of 96% is what you get by approving everything. We report
> PR-AUC and cost-per-1,000-transactions instead.

Runtime: **CPU is fine** (no GPU needed). End-to-end ~3–6 min on Colab.

## 1. Get the code

Point this at your GitHub repo (recommended for the submission). If you haven't pushed
yet, you can instead upload the project folder to Colab and `%cd` into it.

In [ ]:
# Option A — clone from GitHub (edit the URL to your repo):
REPO_URL = ''  # e.g. 'https://github.com/<you>/razorpay-fraud-risk.git'

import os
if REPO_URL:
    !git clone -q $REPO_URL
    %cd razorpay-fraud-risk
else:
    print('No REPO_URL set. Upload the project folder, then %cd into it.')
    print('Current dir:', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q numpy pandas scikit-learn matplotlib joblib

## 3. Connect Kaggle and download the dataset

1. On kaggle.com → your avatar → **Settings** → **API** → *Create New Token*. This
   downloads `kaggle.json`.
2. Go to the competition page **IEEE-CIS Fraud Detection** and click *Join / Late
   Submission* once, to accept the rules (required before the API will let you download).
3. Run the cell below and upload `kaggle.json` when prompted.

In [ ]:
from google.colab import files
import os, json
print('Upload your kaggle.json:')
up = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(list(up.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed.')

In [ ]:
# Download + unzip into ./data (only the two files we use)
!mkdir -p data
!kaggle competitions download -c ieee-fraud-detection -f train_transaction.csv -p data
!kaggle competitions download -c ieee-fraud-detection -f train_identity.csv -p data
!cd data && (unzip -o -q '*.zip' 2>/dev/null; ls -la)

## 4. Run the full pipeline on real data

This trains every model, calibrates it, picks money-optimal thresholds on the
validation window, and reports cost on the **latest** (held-out) time window.

In [ ]:
!python run_pipeline.py --data data/ --outdir reports

## 5. The three plots that tell the story

In [ ]:
from IPython.display import Image, display
for p in ['reports/pr_curve.png', 'reports/calibration.png', 'reports/cost_curve.png']:
    display(Image(p))

In [ ]:
import json
print(json.dumps(json.load(open('reports/summary.json')), indent=2))

## 6. Score live transactions (with reasons)

This is what a payments engineer would actually call: one transaction in, an
**action + calibrated score + the top reasons** out. The last example feeds in
garbage on purpose — the system routes it to manual review instead of crashing.

In [ ]:
import pandas as pd, numpy as np
from config import TARGET, AMOUNT_COL, decide
from src.data import load_ieee_cis, time_based_split
from src.features import Featurizer
from src.model import build_models, PlattCalibrator
from src.evaluate import average_precision, search_cost_thresholds
from src.demo import score_transaction

# rebuild the chosen model in-notebook so we can score interactively
df = load_ieee_cis('data/train_transaction.csv',
                   'data/train_identity.csv' if __import__('os').path.exists('data/train_identity.csv') else None)
sp = time_based_split(df)
fz = Featurizer().fit(sp['train'])
Xtr, ytr = fz.transform(sp['train']), sp['train'][TARGET].values
Xva, yva = fz.transform(sp['valid']), sp['valid'][TARGET].values
models = build_models()
name = 'gradient_boosted' if 'gradient_boosted' in models else 'logistic_numpy'
est = models[name].fit(Xtr.values, ytr)
cal = PlattCalibrator().fit(est.predict_proba(Xva.values)[:,1], yva)
va_amt = sp['valid'][AMOUNT_COL].values
p_va = cal.predict(est.predict_proba(Xva.values)[:,1])
best, _ = search_cost_thresholds(yva, p_va, va_amt)
t_low, t_high = best['t_low'], best['t_high']
feat = list(Xtr.columns)
print(f'Using {name}; thresholds allow<{t_low:.2f}<=review<{t_high:.2f}<=block')

In [ ]:
import json
# a few real transactions from the held-out window + one broken input
samples = [sp['test'].iloc[i].drop(labels=[TARGET]).to_dict() for i in range(3)]
samples.append('THIS IS NOT A TRANSACTION')  # graceful-failure demo
for i, tx in enumerate(samples):
    out = score_transaction(tx, est, fz, cal, t_low, t_high, feat)
    print(f'--- input {i} ---')
    print(json.dumps(out, indent=2, default=str))

## 7. What to say in the video / README

- **Money, not accuracy.** Read the cost-per-1,000 line: our policy vs. approving
  everything vs. blocking everything. That delta is the pitch.
- **Calibrated + explainable.** A score of 0.9 means ~90% (calibration plot), and every
  decision comes with reasons (exact weights for the linear model; an occlusion method
  for the gradient-boosted one -- no SHAP dependency).
- **Honest guardrail.** The threshold search is capped at a realistic manual-review rate,
  so it can't cheat by 'reviewing everything'. See BUILDLOG.md for the night that broke on.
- **Leakage-free.** Time-ordered split; the model is chosen on validation and the test set
  is scored exactly once. It degrades gracefully: garbage in -> routed to review, never a crash.